In [1]:
# Let's peek at the first 10 lines of this massive 1.5GB file without crashing the PC
log_file_path = '../data/HDFS.log'

print("First 10 lines of the raw HDFS Log File:\n")
with open(log_file_path, 'r') as file:
    for i in range(10):
        print(file.readline().strip())

First 10 lines of the raw HDFS Log File:

081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010
081109 203518 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906
081109 203519 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.10.6:40524 dest: /10.250.10.6:50010
081109 203519 145 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.14.224:42420 dest: /10.250.14.224:50010
081109 203519 145 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_-1608999687919862906 terminating
081109 203519 145 INFO dfs.DataNode$PacketResponder: PacketResponder 2 for block blk_-1608999687919862906 terminating
081109 203519 145 INFO dfs.DataNode$PacketResponder: Received block blk_-1608999687919862906 of size 91178 from /10.250.10.6
081109 203519 145

In [2]:
import pandas as pd
import re

log_file_path = '../data/HDFS.log'
csv_output_path = '../data/HDFS_Parsed.csv'

# This Regex pattern tells Python exactly how to slice the log line into columns
log_pattern = re.compile(r'^(\d{6})\s+(\d{6})\s+(\d+)\s+(\w+)\s+(.*?):\s+(.*)$')

parsed_data = []

print("Starting to parse the log file... this might take a minute.")

# Let's just process the first 50,000 lines so it goes fast and we can start training
with open(log_file_path, 'r') as file:
    for i, line in enumerate(file):
        if i >= 50000:  
            break
            
        match = log_pattern.match(line)
        if match:
            date, time, pid, level, component, message = match.groups()
            
            # The most important part of HDFS logs: pulling out the 'Block ID'
            block_id_match = re.search(r'(blk_[-]?\d+)', message)
            block_id = block_id_match.group(1) if block_id_match else "None"
            
            parsed_data.append([date, time, pid, level, component, block_id, message])

# Convert our list into a Pandas DataFrame
columns = ['Date', 'Time', 'PID', 'Level', 'Component', 'BlockId', 'Message']
df = pd.DataFrame(parsed_data, columns=columns)

# Save it as a clean CSV file
df.to_csv(csv_output_path, index=False)

print(f"Done! Successfully parsed {len(df)} lines.")
print(f"Clean CSV saved to: {csv_output_path}")
display(df.head())

Starting to parse the log file... this might take a minute.
Done! Successfully parsed 50000 lines.
Clean CSV saved to: ../data/HDFS_Parsed.csv


,Date,Time,PID,Level,Component,BlockId,Message
0,081109,203518,143,INFO,dfs.DataNode$DataXceiver,blk_-1608999687919862906,Receiving block blk_-1608999687919862906 src: ...
1,081109,203518,35,INFO,dfs.FSNamesystem,blk_-1608999687919862906,BLOCK* NameSystem.allocateBlock: /mnt/hadoop/m...
2,081109,203519,143,INFO,dfs.DataNode$DataXceiver,blk_-1608999687919862906,Receiving block blk_-1608999687919862906 src: ...
3,081109,203519,145,INFO,dfs.DataNode$DataXceiver,blk_-1608999687919862906,Receiving block blk_-1608999687919862906 src: ...
4,081109,203519,145,INFO,dfs.DataNode$PacketResponder,blk_-1608999687919862906,PacketResponder 1 for block blk_-1608999687919...


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

print("1. Loading our parsed logs and the REAL Kaggle labels...")
# Load the logs we parsed earlier
df = pd.read_csv('../data/HDFS_Parsed.csv')

# Load the real labels (Change this filename to whatever it is called in your Kaggle download!)
labels_df = pd.read_csv('../data/anomaly_label.csv') 

print("2. Engineering features and merging with real labels...")
# Group by BlockId to create features
features = df.groupby('BlockId').agg(
    total_logs=('Message', 'count'),
    unique_components=('Component', 'nunique')
).reset_index()

# Merge our features with the real Kaggle labels
# Assuming the label file has columns 'BlockId' and 'Label' (where Label is 'Normal' or 'Anomaly')
merged_df = pd.merge(features, labels_df, on='BlockId', how='inner')

# Convert text labels to numbers (1 for Anomaly/Fault, 0 for Normal)
merged_df['fault_label'] = merged_df['Label'].apply(lambda x: 1 if x == 'Anomaly' else 0)

print("3. Training the Random Forest Model on REAL data...")
X = merged_df[['total_logs', 'unique_components']] 
y = merged_df['fault_label']                       

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Test the Model
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"\n======================================")
print(f"✅ REAL MODEL TRAINING COMPLETE!")
print(f"🎯 Accuracy: {accuracy * 100:.2f}%")
print(f"======================================\n")
print("Detailed Performance Report:")
print(classification_report(y_test, predictions))

1. Loading our parsed logs and the REAL Kaggle labels...
2. Engineering features and merging with real labels...
3. Training the Random Forest Model on REAL data...

✅ REAL MODEL TRAINING COMPLETE!
🎯 Accuracy: 97.56%

Detailed Performance Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       795
           1       0.80      0.17      0.29        23

    accuracy                           0.98       818
   macro avg       0.89      0.59      0.64       818
weighted avg       0.97      0.98      0.97       818



In [6]:
import joblib

# Save the trained model into Dheeraj's backend folder
model_path = '../backend/random_forest_model.pkl'
joblib.dump(model, model_path)

print(f"Model successfully saved to {model_path}!")

Model successfully saved to ../backend/random_forest_model.pkl!
